In [1]:
import sys
sys.path.insert(0,'..')
from warnings import filterwarnings
filterwarnings("ignore")
%load_ext autoreload
%autosave 180

Autosaving every 180 seconds


In [2]:
%autoreload
import os
import random
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from apex import amp
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader
from source.version11.data import trainLoader
from source.version11.model import EfficientModel
from source.version11.train import trainModel
from source.version11.loss import BCELoss
from catalyst.data.sampler import BalanceClassSampler

In [3]:
SEED = 42

def seed(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True
    return None

seed(42)

In [4]:
def train(fold):
    loader = {}
    loader['image_path'] = '../../data/combined/train/train/'
    loader['label_path'] = '../../data/combined/data.csv'
    loader['fold_idx'] = fold
    train, valid = trainLoader(**loader)
    params = {}
    params['batch_size'] = 10
    params['num_workers'] = 3
    params['drop_last'] = True
    sampler = BalanceClassSampler(labels = train.labels(), mode="downsampling")
    train = DataLoader(train, shuffle=True, **params)
    valid = DataLoader(valid, **params)
    model = EfficientModel()
    model = model.to('cuda:0')
    optimizer = AdamW(model.parameters(), lr=3e-05, weight_decay=0.)
    schedular = ReduceLROnPlateau(optimizer, factor=0.8, patience=0, min_lr=1e-8)
    model, optimizer = amp.initialize(model, optimizer, opt_level='O2', verbosity=False)
    trainer = {}
    trainer['model'] = model
    trainer['train_data'] = train
    trainer['valid_data'] = valid
    trainer['loss_fn'] = BCELoss()
    trainer['optimizer'] = optimizer
    trainer['save_path'] = '../../model/version11/model_{}.pt'.format(fold)
    trainer['epochs'] = 15
    trainer['batch'] = 10
    trainer['scheduler'] = schedular
    trainModel(**trainer)
    model.cpu()
    del model
    return None

In [ ]:
train(0)

Train Images: 35557 Valid Images: 6478
Loaded pretrained weights for efficientnet-b5


100% 35550/35550 [29:06<00:00, 20.35it/s, trn_ls=0.4659, val_ls=0.1888, val_mt=0.8740]
100% 35550/35550 [29:01<00:00, 20.41it/s, trn_ls=0.3179, val_ls=0.1614, val_mt=0.8857]
100% 35550/35550 [29:04<00:00, 20.38it/s, trn_ls=0.2482, val_ls=0.1515, val_mt=0.9279]
 29% 10460/35550 [08:13<20:02, 20.87it/s, trn_ls=0.23360]IOPub message rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_msg_rate_limit`.

Current values:
NotebookApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
NotebookApp.rate_limit_window=3.0 (secs)

100% 35550/35550 [29:10<00:00, 20.31it/s, trn_ls=0.2311, val_ls=0.1644, val_mt=0.9149]
 18% 6510/35550 [05:08<22:29, 21.52it/s, trn_ls=0.21710]IOPub message rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_msg_rate_limit`.

In [ ]:
train(1)

In [ ]:
train(2)

In [ ]:
train(3)

In [ ]:
train(4)